# To impliment Convolution, depth seperable convolution, de-convolution operations and their advantages across various deep learning architectures.

# Execute and show output before leaving lab to get continuous lab evalutation marks.

In [1]:
import numpy as np
import time

In [2]:
# Input feature map - Initial
# Height Width Channels
H, W, M = 32, 32, 64 # Experiment with H - 56,112,14
# (Number of output filters)
N = 128
K = 3 #kernel size of filter 3 by 3
# Ignored Stride and padding
# VGG/ResNet/Inception - Normal conv
# EffieientNet/MobileNet - Depth seperable conv

In [3]:
# 3 x 3 filter 128 filters i will convolve with 32 x 32 images 64 images

In [5]:
# Generate feature maps randomly
input_feature_map = np.random.randn(H,W,M)
input_feature_map[:,:,1] # To show contents of feature map
input_feature_map[:,:,1].shape # Size of each feature
input_feature_map.shape # Input feature shape

(32, 32, 64)

In [10]:
# Generate filters randomly
standard_filters = np.random.randn(K,K,M,N)
standard_filters.shape

(3, 3, 64, 128)

In [11]:
output = np.zeros((H, W, N))
output.shape

(32, 32, 128)

In [12]:
H_out, W_out = H, W
# Conv2D(3,64,input) sum(x(k)*(Multiply)h(n-k)(shift))

In [13]:
start_time = time.time()
output = np.zeros((H_out, W_out, N))
# Apply convolution filter to each output channel
for n in range(N):  # For each filter 128
    for c in range(M):  # For each input channel 64
        for i in range(H_out - K + 1):  # Iterate over height positions
            for j in range(W_out - K + 1): # Iterate over width positions
                output[i, j, n] += np.sum(
                    input_feature_map[i:i+K, j:j+K, c] * standard_filters[:, :, c, n]
                )
standard_time = time.time() - start_time
#print(output)

In [14]:
standard_time

46.92196536064148

In [16]:
# Generate random depthwise and pointwise filters 3 3 64 128
depthwise_filters = np.random.randn(K, K, M, 1) # 3 3 64 1
pointwise_filters = np.random.randn(1, 1, M, N) # 1 1 64 128

In [17]:
depthwise_output = np.zeros((H_out, W_out, M))
# Apply Depthwise Convolution (each channel has its own filter)
start_time = time.time()
for c in range(M):
    for i in range(H_out - K + 1):
        for j in range(W_out - K + 1):
            depthwise_output[i, j, c] = np.sum(
                input_feature_map[i:i+K, j:j+K, c] * depthwise_filters[:, :, c, 0]
            )

# Apply Pointwise Convolution (1x1 Conv)
output = np.zeros((H_out, W_out, N))
for n in range(N):
    output[:, :, n] = np.sum(depthwise_output * pointwise_filters[:, :, :, n], axis=2)
standard_time = time.time() - start_time

In [20]:
depthwise_filters.shape

(3, 3, 64, 1)

In [21]:
pointwise_filters.shape

(1, 1, 64, 128)

In [22]:
depthwise_output = np.zeros((H, W, M))
depthwise_output.shape

(32, 32, 64)

In [23]:
standard_time

0.37652158737182617

In [24]:
# Compute total number of operations
standard_ops = K * K * M * N * H * W
depthwise_ops = (K * K * M * H * W) + (M * N * H * W)

In [25]:
print(f"Total Operations in Standard Convolution: {standard_ops:,}")
print(f"Total Operations in Depthwise Separable Convolution: {depthwise_ops:,}")
print(f"Computational Reduction: {100 * (1 - depthwise_ops / standard_ops):.2f}%")

Total Operations in Standard Convolution: 75,497,472
Total Operations in Depthwise Separable Convolution: 8,978,432
Computational Reduction: 88.11%


In [26]:
# Deconvolution (Transposed Convolution) 7 7 512 - 7 7 1024

In [45]:
# Generate random filters for deconvolution
deconv_filters = np.random.randn(K, K, N, M)
output_size = M

In [46]:
H_out, W_out = W, H
C_out = output_size  # Output channels should match M
output = np.zeros((H_out, W_out, C_out))  # Ensure output shape is correct
# Iterate over input channels (N) first, then map to output channels (M)
for n in range(N):  # Iterate over input channels (from previous layer)
    for c in range(C_out):  # Iterate over output channels (M)
        for i in range(H_out - K + 1):  # Fix: Use H, not H_out
            for j in range(W_out - K + 1):  # Fix: Use W, not W_out
                output[i:i+K, j:j+K, n] += input_feature_map[i, j, c] * deconv_filters[:, :, c, n]

IndexError: index 64 is out of bounds for axis 2 with size 64